##  Setup & Configuration

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

In [3]:
DATA_DIR        = Path("Data")
END_DATE        = "2026-01-31"
DELAY_THRESHOLD = 5
WEATHER_CACHE   = DATA_DIR / "weather_toronto.csv"
EXPORT_PATH     = DATA_DIR / "bus_powerbi_1.csv"
EXPORT_MODEL    = DATA_DIR / "bus_model_export_1.csv"

## Loading Raw Data

Notice that the raw data consists of some inconsistencies between 2022-2024 and 2025+. The differences are noted below:

| | 2022-2024 XLSX | 2025+ CSV |
|---|---|---|
| Route | `Route` : integer (mostly) | `Line` : string e.g. "102 MARKHAM ROAD" |
| Location | `Location` : intersection name | `Station` : stop name |
| Incident type | `Incident` : plain English (15 categories) | `Code` : bus-specific code e.g. "EFO", "MFSH" |
| Direction | `Direction` : N/S/E/W + junk | `Bound` : N/S/E/W + junk |

ALso, the Code Descriptions file, `Bus Code Descriptions.csv` has a Windows-1252 encoding artefact: `â` should be read as `-` (em-dash corruption).

In [4]:
df_bus_codes = pd.read_csv(DATA_DIR / "Bus Code Descriptions.csv")
# Fix encoding artefact: the 3-character sequence U+00E2 U+0080 U+0093 is a
# double-encoded en-dash (–). We must replace all three chars together in one go
# replacing just the first char leaves two invisible control chars behind.
CORRUPT = "â"
df_bus_codes["DESCRIPTION"] = (
    df_bus_codes["DESCRIPTION"]
    .str.replace(CORRUPT, " - ", regex=False)
    .str.strip()
)
CODE_TO_DESC = dict(zip(df_bus_codes["CODE"], df_bus_codes["DESCRIPTION"]))
print(f"{len(df_bus_codes)} bus delay codes")
pd.set_option("display.max_rows", None)
print(df_bus_codes[["CODE","DESCRIPTION"]].to_string(index=False))

46 bus delay codes
 CODE                                   DESCRIPTION
  EFB                                          BODY
EFCAN                                  CANCELLATION
  EFD                                         DOORS
 EFDB                                   DISC BRAKES
 EFHV                                  HIGH VOLTAGE
EFHVA                              HEAT / VENT / AC
 EFLV                                   LOW VOLTAGE
  EFO                                         OTHER
  EFP                                    PROPULSION
 EFRA                                   RAMP ISSUES
  EFT                                        TRUCKS
 EFTB                                  TRACK BRAKES
 MFCN                                      CLEANING
 MFDV                                  ON DIVERSION
MFESA              NO OPERATOR AVAILABLE DUE TO ESA
 MFFD                                  FARE DISPUTE
 MFLD                                LABOUR DISPUTE
  MFO                                        

In [6]:
# loading xslx files (2022-2024)
print("Loading 2022-2024 XLSX files...")
xlsx_sources = [
    (DATA_DIR / "ttc-bus-delay-data-2022.xlsx", "2022"),
    (DATA_DIR / "ttc-bus-delay-data-2023.xlsx", "2023"),
    (DATA_DIR / "ttc-bus-delay-data-2024.xlsx", "2024"),
]
dfs_xlsx = [pd.read_excel(p, sheet_name=s) for p, s in xlsx_sources]
df_xlsx = pd.concat(dfs_xlsx, ignore_index=True)
print(f"  2022-2024 combined : {len(df_xlsx):,} rows")
print(f"  Columns : {df_xlsx.columns.tolist()}")
print(df_xlsx.head(3).to_string())

Loading 2022-2024 XLSX files...
  2022-2024 combined : 174,557 rows
  Columns : ['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']
        Date Route   Time       Day                Location               Incident  Min Delay  Min Gap Direction  Vehicle
0 2022-01-01   320  02:00  Saturday        YONGE AND DUNDAS          General Delay          0        0       NaN     8531
1 2022-01-01   325  02:00  Saturday  OVERLEA AND THORCLIFFE              Diversion        131      161         W     8658
2 2022-01-01   320  02:00  Saturday       YONGE AND STEELES  Operations - Operator         17       20         S        0


In [7]:
# loading 2025+ csv file
df_csv = pd.read_csv(DATA_DIR / "TTC Bus Delay Data since 2025.csv")
df_csv = df_csv.drop(columns=["_id"], errors="ignore")
print(f"  2025+ rows : {len(df_csv):,} rows")
print(f"  Columns : {df_csv.columns.tolist()}")
print(df_csv.head(3).to_string())

  2025+ rows : 69,037 rows
  Columns : ['Date', 'Line', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle']
                  Date              Line   Time        Day            Station   Code  Min Delay  Min Gap Bound  Vehicle
0  2025-01-01T00:00:00  102 MARKHAM ROAD  02:15  Wednesday     WARDEN STATION  MFESA         20       40     N     3442
1  2025-01-01T00:00:00     65 PARLIAMENT  02:15  Wednesday    KIPLING STATION   MFUS          0        0   NaN        0
2  2025-01-01T00:00:00           64 MAIN  02:40  Wednesday  BROADVIEW STATION   MFUI          0        0   NaN     8546


## Data Normalization & Cleaning

### Approach
- Handling Route inconsistencies : extract leading integer from `Route`/`Line`; drop non-service rows (garage, training, RAD/OTC internal runs)
-  Column renaming : `Location` → `Stop`, `Station` → `Stop`, `Direction` → `Bound`
-  Code descriptions : merge 2025+ codes with `Bus Code Descriptions.csv` to get real `Description`. for 2022-2024, use `Incident` text directly
-  Typo / leaked-code handling : ~25 rows in the 2025+ CSV contain subway codes (MUIE, MUIS, SUDP, etc.) that don't belong in the bus dataset; treat as "Unknown"
-  Bound normalization : keep only {N, S, E, W}, null everything else
-  Broad category assignment : map descriptions → `Code_Category'

In [8]:
# Route is usually an integer but some rows have strings: "927 HIGHWAY 27", "RAD", "OTC"
# Extract the leading digits, rows without a numeric route are non-service.
df_xlsx["Route"] = (
    df_xlsx["Route"].astype(str).str.strip()
    .str.extract(r"^(\d+)", expand=False) # take only leading digits
    .astype(float) # converto to float since NaN can't exist in an int
)
n_before = len(df_xlsx)
df_xlsx = df_xlsx.dropna(subset=["Route"]).copy()
df_xlsx["Route"] = df_xlsx["Route"].astype(int)
print(f"2022-2024 — dropped {n_before - len(df_xlsx):,} non-service rows (RAD, OTC, etc.)")


2022-2024 — dropped 1,885 non-service rows (RAD, OTC, etc.)


In [9]:
# Rename to unified column names
df_xlsx = df_xlsx.rename(columns={"Location": "Stop", "Direction": "Bound"})

# For 2022-2024, Incident is the description. Code is unavailable.
# Description column = Incident text; Code column = None
df_xlsx["Code"]        = None
df_xlsx["Description"] = df_xlsx["Incident"]
df_xlsx["Line_Name"]   = None


In [ ]:
# notice that there are some data leakeage between the bus data and subway data
# e.g "YU", "Line 1", etc.
# drop non-service Line values: "LINE 1 SHUTTLE", "RAD", "SRT", subway line names, etc.
NON_SERVICE_RE = (
    r"^LINE\b|^RAD\b|^SRT\b|^YU\b|^BD\b|^SHP\b"
    r"|^SHUTTLE\b|^FLEET\b|^TRAINING\b|^OTC\b|^A\b"
)
non_svc = df_csv["Line"].str.upper().str.contains(NON_SERVICE_RE, regex=True, na=False)
n_before = len(df_csv)
df_csv = df_csv[~non_svc].copy()
print(f"2025+ CSV — dropped {n_before - len(df_csv):,} non-service Line rows")

# extract route number: "102 MARKHAM ROAD" to 102
df_csv["Route"] = df_csv["Line"].str.extract(r"^(\d+)", expand=False).astype(float)
n_before = len(df_csv)
df_csv = df_csv.dropna(subset=["Route"]).copy()
df_csv["Route"] = df_csv["Route"].astype(int)
print(f"2025+ CSV — dropped {n_before - len(df_csv):,} further unresolvable rows")

# keeep the route name part for reference
df_csv["Line_Name"] = (
    df_csv["Line"].str.extract(r"^\d+\s+(.*)", expand=False).str.strip()
)

# rename Station to Stop
df_csv = df_csv.rename(columns={"Station": "Stop"})

2025+ CSV — dropped 244 non-service Line rows
2025+ CSV — dropped 851 further unresolvable rows


In [11]:
# merge bus code descriptions
# subway codes won't match the bus lookup and get Description = "Unknown".
df_csv["Description"] = df_csv["Code"].map(CODE_TO_DESC).fillna("Unknown")

unknown_cnt = (df_csv["Description"] == "Unknown").sum()
print(f"2025+ CSV — {unknown_cnt} rows with unrecognised codes (likely subway code leakage → 'Unknown')")
if unknown_cnt > 0:
    print("  Unrecognised codes:", df_csv.loc[df_csv["Description"]=="Unknown","Code"].value_counts().to_dict())

# for 2025+, derive Incident from Description (mirrors 2022-2024 Incident usage)
df_csv["Incident"] = df_csv["Description"]

# data concatenation
COMMON_COLS = ["Date", "Time", "Day", "Route", "Stop", "Line_Name",
               "Code", "Incident", "Description", "Min Delay", "Min Gap", "Bound", "Vehicle"]
df = pd.concat([df_xlsx[COMMON_COLS], df_csv[COMMON_COLS]], ignore_index=True)

print(f"Total shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range : {pd.to_datetime(df['Date']).min().date()} to {pd.to_datetime(df['Date']).max().date()}")

2025+ CSV — 22 rows with unrecognised codes (likely subway code leakage → 'Unknown')
  Unrecognised codes: {'MTDV': 3, 'TTPD': 3, 'MTO': 2, 'MTNOA': 2, 'MTSAN': 2, 'MUO': 2, 'SUDP': 1, 'MTVIS': 1, 'STO': 1, 'MUIS': 1, 'MTUS': 1, 'MUIE': 1, 'SRO': 1, 'ETO': 1}
Total shape : 240,614 rows x 13 columns
Date range : 2022-01-01 to 2026-01-31


In [12]:
# normalize Bounds / Direction values to N/S/E/W, set invalid values to None
VALID_BOUNDS = {"N", "S", "E", "W"}
df["Bound"] = df["Bound"].where(df["Bound"].isin(VALID_BOUNDS), other=None)
print(f"Bound distribution after normalization:")
print(df["Bound"].value_counts().to_string())
print(f"Bound null %: {df['Bound'].isna().mean():.1%}  (kept for Power BI, not used in model)")
print()

# assign code category
# works on both 2022-2024 Incident text and 2025+ Description text.

def assign_bus_category(text):
    t = str(text).lower()
    # Mechanical / Equipment
    if any(w in t for w in ["body", "door", "brake", "voltage", "propulsion",
                             "trucks", "ramp", "heat", "vent", "ac", "mechanical",
                             "collision - ttc"]):
        return "Mechanical"
    # Passenger / Medical
    if any(w in t for w in ["ill customer", "on board injury", "transported",
                             "transportation declined"]):
        return "Passenger / Medical"
    # Security / Safety
    if any(w in t for w in ["assault", "disorderly", "bomb", "police", "suspicious",
                             "security", "fare dispute", "investigation"]):
        return "Security / Safety"
    # Cleaning / Sanitary
    if any(w in t for w in ["cleaning", "unsanitary", "disinfection", "sanitary"]):
        return "Cleaning / Sanitary"
    # Emergency / Traffic / Collision
    if any(w in t for w in ["fire", "smoke", "collision", "personal injury",
                             "property damage", "emergency", "road blocked"]):
        return "Emergency / Traffic"
    # Operational Disruption (diversion, shuttle, parade, can't hold schedule)
    if any(w in t for w in ["diversion", "shuttle bus", "parade", "march",
                             "general delay", "held by", "cancellation"]):
        return "Operational Disruption"
    # Operations (operator, schedule, late, vision)
    if any(w in t for w in ["operator", "schedule", "late entering", "late vehicle",
                             "switch", "operations", "vision", "no operator available",
                             "utilized off route", "utilized off", "labour", "labor"]):
        return "Operations"
    # Weather
    if any(w in t for w in ["weather", "snow", "ice", "flood"]):
        return "Weather"
    return "Other"

df["Code_Category"] = df["Description"].apply(assign_bus_category)

print("Code_Category distribution:")
print(df["Code_Category"].value_counts().to_string())
print()
print(f"Final shape after unification: {df.shape[0]:,} rows x {df.shape[1]} columns")

Bound distribution after normalization:
Bound
N    55199
S    52714
E    47336
W    44280
Bound null %: 17.1%  (kept for Power BI, not used in model)

Code_Category distribution:
Code_Category
Mechanical                74698
Operations                69795
Operational Disruption    31476
Security / Safety         19230
Other                     17578
Emergency / Traffic       14620
Cleaning / Sanitary        7919
Passenger / Medical        4157
Weather                    1141

Final shape after unification: 240,614 rows x 14 columns


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240614 entries, 0 to 240613
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Date           240614 non-null  object
 1   Time           240614 non-null  str   
 2   Day            240614 non-null  str   
 3   Route          240614 non-null  int64 
 4   Stop           240613 non-null  str   
 5   Line_Name      67150 non-null   object
 6   Code           67942 non-null   object
 7   Incident       240614 non-null  str   
 8   Description    240614 non-null  str   
 9   Min Delay      240614 non-null  int64 
 10  Min Gap        240614 non-null  int64 
 11  Bound          199529 non-null  str   
 12  Vehicle        240614 non-null  int64 
 13  Code_Category  240614 non-null  str   
dtypes: int64(4), object(3), str(7)
memory usage: 25.7+ MB


In [14]:
df.head()

,Date,Time,Day,Route,Stop,Line_Name,Code,Incident,Description,Min Delay,Min Gap,Bound,Vehicle,Code_Category
0,2022-01-01 00:00:00,02:00,Saturday,320,YONGE AND DUNDAS,None,None,General Delay,General Delay,0,0,NaN,8531,Operational Disruption
1,2022-01-01 00:00:00,02:00,Saturday,325,OVERLEA AND THORCLIFFE,None,None,Diversion,Diversion,131,161,W,8658,Operational Disruption
2,2022-01-01 00:00:00,02:00,Saturday,320,YONGE AND STEELES,None,None,Operations - Operator,Operations - Operator,17,20,S,0,Operations
3,2022-01-01 00:00:00,02:07,Saturday,320,YONGE AND STEELES,None,None,Operations - Operator,Operations - Operator,4,11,S,0,Operations
4,2022-01-01 00:00:00,02:13,Saturday,320,YONGE AND STEELES,None,None,Operations - Operator,Operations - Operator,4,8,S,0,Operations


### Post Cleaning Summary
~240,000 bus delay incidents from **January 2022 to January 2026**, combining four source files:
- 3 XLSX files (2022, 2023, 2024)
- 1 CSV file (2025+)

| Concept | How to find it |
|---|---|
| **Bus route** | `Route` column : always a clean integer (e.g. `38`). This is the only consistent identifier across all years. |
| **Route name** | `Line_Name` column : e.g. `"HIGHLAND CREEK"`. **Only populated for 2025+ rows**; null for 2022–2024 because that data never included names. To get "38 Highland Creek", combine `Route` + `Line_Name`. |
| **Bus stop / intersection** | `Stop` column : the stop name or intersection (e.g. `"YONGE AND DUNDAS"`) |
| **What caused the delay** | `Description` column : plain English for all rows (e.g. `"DISORDERLY PATRON"`, `"DOORS"`, `"ON DIVERSION"`) |
| **Broad delay category** | `Code_Category` column : groups descriptions into ~8 buckets: Mechanical, Security/Safety, Operational Disruption, etc. **Fully populated for all 240K rows across all years.** |
| **Raw bus code** | `Code` column : the official code abbreviation (e.g. `"SFDP"`, `"EFD"`). **Only populated for 2025+ rows**; null for 2022–2024 because those years used plain-text incidents instead. |
| **Direction** | `Bound` : N/S/E/W only. ~17% null (invalid values like "NB", "SB" were nulled out). |



## EDA